In [10]:
#读取数据，dataset和dataloader
'''
dataset提供获取数据和label
dataloader为网络提供不同的数据形式
'''
from torch.utils.data import Dataset
#读取图片
from PIL import Image
import os
class MyData(Dataset):
    #数据初始化
    def __init__(self,root_dir,label_dir):
        self.root_dir = root_dir
        self.label_dir = label_dir
        #合并地址
        self.path = os.path.join(self.root_dir,self.label_dir)
        #获得该路径下所有文件夹的列表
        self.img_path = os.listdir(self.path)
    #返回每一张图片
    def __getitem__(self, index):
        #获得图片在文件夹下的路径
        img_name = self.img_path[index]
        img_item_path = os.path.join(self.root_dir,self.label_dir,img_name)
        img = Image.open(img_item_path)
        label = self.label_dir
        return img,label
    def __len__(self):
        return len(self.img_path)
root_dir = 'D:/miniconda/internshipproject/.vscode/smalltudui_pytorch/练手数据集(1)/hymenoptera_data/hymenoptera_data/train'
ants_label_dir = 'ants'
ants_dataset = MyData(root_dir,ants_label_dir)
bees_label_dir = 'bees'
bees_dataset = MyData(root_dir,bees_label_dir)
img,label = bees_dataset[0]
img.show()

In [ ]:
#tensorboard的使用
import numpy as np
from PIL import Image
from torch.utils.tensorboard import SummaryWriter
#创建summarywriter实例，制定文件夹
#   writer可以接收numpy，tensor数据类型
writer = SummaryWriter('logs')
#纪录输入图像
img_PIL = Image.open('D://miniconda//internshipproject//.vscode//smalltudui_pytorch//练手数据集(1)//练手数据集//train.png')
#将4channel转为3channel
img_PIL = img_PIL.convert("RGB")
Image_array = np.array(img_PIL)
#add_image接收array和tensor数据类型
#数据类型是HWC
writer.add_image('test',Image_array,1,dataformats='HWC')
#纪律标量，如loss
for i in range(100):
    writer.add_scalar("y=x",i,i)
#打开tensorboard tensorboard --logdir=smalltudui_pytorch/logs

In [ ]:
#数据转换transforms,裁切，转换成tensor数据类型
from torchvision import transforms
#创建totensor类
#totensor会将hwc转为chw类型
tensor_trans = transforms.ToTensor()
PIL_tensor = tensor_trans(img_PIL)
import cv2
#cv读不出中文路径
#cv_img = cv2.imread('D://miniconda//internshipproject//.vscode//smalltudui_pytorch//练手数据集(1)//练手数据集//train//ants_image//0013035.jpg')
#cv_tensor = tensor_trans(cv_img)
#__call__函数就是直接被调用的函数
'''
class a():
    def __call__(self,b):
        print(b)
这样直接使用创建的实例就是用call函数
'''
#归一化tansforms中normalize类
trans_norm = transforms.Normalize([0.5,0.5,0.5],[0.5,0.5,0.5])
img_norm = trans_norm(PIL_tensor)
writer.add_image('test',img_norm,2,dataformats='CHW')

In [29]:
#resize类的使用
#  resize接收pil image和tensor数据类型
trans_resize = transforms.Resize((512,512))
img_resize = trans_resize(img_PIL)
img_resize = tensor_trans(img_resize)
writer.add_image("Resize",img_resize,0)
#  compose函数，compose函数的作用是将
#    多个过程放到一块进行
trans_resize_2 = transforms.Resize(512)
trans_compose = transforms.Compose([tensor_trans,trans_resize_2])
img_resize_2 = trans_compose(img_PIL)
writer.add_image("Resize",img_resize_2,1)
trans_crop = transforms.RandomCrop(224)
trans_compose_2 = transforms.Compose([tensor_trans,trans_crop])
for i in range(10):
    img_crop = trans_compose_2(img_PIL)
    writer.add_image('RandomCrop',img_crop,i)

In [30]:
#dataset和transforms的联合使用
import torchvision
#数据集里是pilimage数据类型，转换为tensor数据类型
dataset_transform = transforms.Compose([
    transforms.ToTensor()
])
train_set = torchvision.datasets.CIFAR10(root='./cifar-10-python',train=True,download=True,transform=dataset_transform)
test_set = torchvision.datasets.CIFAR10(root='./cifar-10-python',train=False,download=True,transform=dataset_transform)

In [39]:
#dataloader的使用
from torch.utils.data import DataLoader
test_loader = DataLoader(dataset = test_set,
                         shuffle = True,
                         #shuffle是是都打乱
                         batch_size = 64,
                         num_workers = 0,
                         #num_workers是选择进程数
                         drop_last = False)
step = 0
for data in test_loader:
    img,target = data
#这里要用add_images不然只能添加一张
    writer.add_images('CIFR10',img,step)
    step+=1

![神经网络结构图](./神经网络结构图.png)

In [45]:
#模型实例
import torch
import torch.nn as nn
from torch.nn import Flatten,Conv2d, MaxPool2d, Linear
class kk(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = Conv2d(3,32,5,padding = 2)
        self.maxpool1 = MaxPool2d(2)
        self.conv2 = Conv2d(32,32,5,padding=2)
        self.maxpool2 = MaxPool2d(2)
        self.conv3 = Conv2d(32,64,5,padding=2)
        self.maxpool3 = MaxPool2d(2)
        self.flatten = Flatten()
        self.linear1 = Linear(1024,64)
        self.linear2 = Linear(64,10)
    def forward(self,x):
        x = self.maxpool3(self.conv3(self.maxpool2(self.conv2(self.maxpool1(self.conv1(x))))))
        x = self.linear1(self.flatten(x))
        output = self.linear2(x)
        return output
'''
              |
              |
        与sequtial等价
    def __init(self):
        super().__init()
        self.model = Sequential(
            Conv2d(3,32,5,padding = 2),
            MaxPool2d(2),
            Conv2d(32,32,5,padding=2),
            MaxPool2d(2),
            Conv2d(32,64,5,padding=2),
            MaxPool2d(2),
            Flatten(),
            Linear(1024,64),
            Linear(64,10)
            )
    def forward(self,x):
        x = self.model(x)
        return x

'''
kk = kk()
input = torch.ones((64,3,32,32))
writer.add_graph(kk,input)

In [ ]:
#卷积层
#  卷积层也是线性的，1*1卷积层相当于一个全连接层
#  全连接层会损失空间信息，而1*1卷积层不会
#   对于32*32*64的输入，输出为32*32*128
#   1*1卷积层相当于在每一个空间位置上
#    都执行一个 Linear(64,128) 的全连接层
'''
nn.Conv2d(
    input,#输入（batchsize，inchannel，h，w）
    kernel#核大小（outchannel,inchannel/groups,h,w)
    in_channels,#输入通道数
    out_channels,#输出通道数
    kernel_size,#卷积核大小
    stride=1,#步长
    padding=0,#边缘伸出去长度
    dilation=1,
    groups=1,#分组卷积，默认值为1，
             #  每个卷积核都有inchannel个2维
             #  卷积核
    bias=True#偏置
)
'''
#池化层
#  池化层不具有可学习参数
#  池化不会改变channel数量，只改变H、W
#  最大池化:在一个池化窗口内选择最大值，保留最显著特征
#  平均池化:在一个池化窗口内求平均值，保留整体信息
'''
nn.MaxPool2d(
    input#输入与卷积层一致
    kernel_size,#池化窗口大小，int只有一个
    stride=None,#步长
    padding=0,#边缘填充大小
    dilation=1,
    return_indices=False,
    ceil_mode=False,#输出尺寸计算时是否向上取整
                    # True就是只有全部覆盖数据才要
)
'''
'''
激活函数
y = relu(x),inplace=False/True，是否原地操作
y = sigmod(x)对于输入的形状不限制
'''
'''
线性层
Linear(in_feature,out_feature,bias)
bias指的是偏置
flatten(x,dim = )#start_dim = 1代表保留0维
                          从一维展开
                end_dim表示从哪里停止展开                

'''

In [ ]:
#模型保存的方式
'''
方法1
torch.save(model,path)
torch.load(path)
方法2
    需要创建结构一致的模型
torch.save(model.state_dict(),path)
model.load_state_dict(torch.load(path))
'''

In [48]:
train_loader = DataLoader(dataset = train_set,
                         shuffle = True,
                         batch_size = 64,
                         drop_last = False)
#损失函数，交叉熵
loss_fn = nn.CrossEntropyLoss()
#优化器，SGD
learning_rate = 0.01
optimizer = torch.optim.SGD(kk.parameters(),lr = learning_rate)

total_train_step = 0
total_test_step = 0
epoch = 10
for i in range(epoch):
    print('-------epoch{}------'.format(i+1))
    #训练步骤

      #对于dropout和normlize层设置状态有效
    kk.train()
    for data in train_loader:
        imgs,targets = data
        output = kk(imgs)
        loss = loss_fn(output,targets)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_train_step+=1
        if total_train_step % 100 == 0:
            print('训练次数{},loss{}'.format(total_train_step,loss))
            writer.add_scalar('train_loss',loss,total_train_step)

    now_size = 0
    current = 0
    #测试步骤
    with torch.no_grad():
        for data in test_loader:
            img_test,target_test = data
            output_test = kk(img_test)
            preds = output_test.argmax(1)
            current += (preds == target_test).sum()
            now_size += target_test.size(0)
        accuracy = current/now_size
        print('accuracy{}'.format(accuracy))
        writer.add_scalar('accuracy',accuracy,total_test_step)
        total_test_step += 1
writer.close()


-------epoch1------
训练次数100,loss1.6740193367004395
训练次数200,loss1.7586625814437866
训练次数300,loss1.5897492170333862
训练次数400,loss1.5578323602676392
训练次数500,loss1.3654558658599854
训练次数600,loss1.5239858627319336
训练次数700,loss1.8385697603225708
accuracy0.42170000076293945
-------epoch2------
训练次数800,loss1.4974865913391113
训练次数900,loss1.5424467325210571
训练次数1000,loss1.5830280780792236
训练次数1100,loss1.645038366317749
训练次数1200,loss1.487953782081604
训练次数1300,loss1.4559786319732666
训练次数1400,loss1.551728367805481
训练次数1500,loss1.2498537302017212
accuracy0.4235000014305115
-------epoch3------
训练次数1600,loss1.453864336013794
训练次数1700,loss1.246374249458313
训练次数1800,loss1.3368310928344727
训练次数1900,loss1.4936943054199219
训练次数2000,loss1.481197714805603
训练次数2100,loss1.5205256938934326
训练次数2200,loss1.3660070896148682
训练次数2300,loss1.5737782716751099
accuracy0.4490000009536743
-------epoch4------
训练次数2400,loss1.538237452507019
训练次数2500,loss1.253989815711975
训练次数2600,loss1.3161026239395142
训练次数2700,loss1.33550035